# Corbes en U del consum de Barcelona: núvol de punts, isòcrones i isotropes
### Notebook v0.1 · línia nova: mapa de l'illa de calor a partir del consum (26/10/2026)

**Pregunta de fons (idea de l'Aleix, 26/10):** hi ha una **corba única** $E=f(T_\text{aire})$ per a Barcelona? Cada codi postal sentiria $T_i = T_\text{Prat} + \Delta T_i$. Si portem totes les corbes sobre la mateixa, el desplaçament $\Delta T_i$ és un mapa de l'illa de calor de l'aire.

**Aquest notebook encara no ajusta res.** Només **mira** les dades, que és el primer pas:
1. la temperatura de referència, **mesurada a l'aeroport del Prat**, sense cap model;
2. el núvol de punts consum–temperatura de la ciutat, separant els dies feiners dels caps de setmana i festius;
3. les **isòcrones** (mateix temps, és a dir, mateixa franja horària o mateix any) de diverses maneres;
4. les **isotropes** (mateixa franja, cada codi postal per separat).

**Vocabulari (acordat a Seguiment el 26/10):**
- **Isotropes:** mateixa franja horària, diferents CP. Si l'illa de calor existeix, s'han de veure desplaçades horitzontalment.
- **Isòcrones:** mateix lloc, diferents franges o anys.

**Com fer-lo servir:** executa les cel·les **d'una en una** i llegeix el text de cada secció abans. Al final hi ha unes preguntes per comprovar que s'entén què surt.

## 1. Instal·lació i paràmetres

**Què fa:** instal·la el paquet de la tesi (només per llegir els CSV del consum i per a dues comprovacions de qualitat) i fixa els paràmetres. Pots canviar qualsevol paràmetre i tornar a executar.

In [ ]:
!pip install -q holidays "git+https://github.com/PauBuchacaMartin/power_temperature_urban@main"

import io, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import holidays
import requests
import consum_temperatura as ct
from consum_temperatura import figstyle as fs
fs.apply_style()

P = {
    "carpeta": "/content/drive/MyDrive/dades_tesi",
    "anys": [2021, 2022, 2023, 2024],     # el 2025 no passa la regla de qualitat (v0.14)
    "anys_cp": [2021, 2022],              # per CP: abans de les recodificacions de la tardor del 2023 (alerta 1)
    "sector": "Residencial",              # prova també "Total" (tots els sectors)
    "estacio_isd": "08181099999",         # Barcelona / El Prat a la base ISD de la NOAA (cal comprovar el nom, secció 2)
    "amplada_bin": 1.0,                   # °C: amplada de cada interval de temperatura de les corbes
    "min_dies_bin": 5,                    # un interval amb menys dies no es dibuixa
    "franja_isotropes": "00-06",          # franja per a les isotropes (de nit, l'illa de calor és més forta)
}
os.makedirs("figures_u01", exist_ok=True)
print("[OK] paràmetres:", P)

## 2. Temperatura de l'aeroport del Prat, «pura i dura»

**D'on surt:** de la base **ISD** (*Integrated Surface Database*) de la NOAA, que guarda les observacions horàries dels aeroports, els METAR. L'estació del Prat té l'indicatiu OMM 08181. A l'ISD el fitxer es diu `08181099999.csv`, un per any. No cal clau. **No ho he pogut comprovar des d'aquí**, perquè el meu entorn no pot accedir a la NOAA. Per això la cel·la comprova el nom de l'estació i s'atura si no és Barcelona.

**Com és la dada en brut:** la columna `TMP` val, per exemple, `+0215,1`:
- `+0215` és la temperatura en dècimes de grau: 21,5 °C;
- `1` és el codi de qualitat;
- `+9999` vol dir que falta la dada.

**Què fa la cel·la, pas a pas:**
1. Baixa un fitxer per any (i el desa al Drive: la segona vegada ja no el baixa).
2. Tradueix `TMP` a °C i llença les dades que falten o que la NOAA marca com a dolentes.
3. Passa l'hora de UTC a l'hora de Barcelona. Les franges del consum són en hora local.
4. Fa la mitjana de cada hora (hi pot haver més d'una observació per hora).
5. Calcula la **mitjana diària** (només si hi ha ≥ 20 hores) i la mitjana de cada **franja** de 6 hores (només si hi ha ≥ 4 hores).

**Per què el Prat:** és a la costa, fora del teixit urbà, i serveix de **referència rural** per a tota la ciutat. Un barri més calent que el Prat ha de consumir com si hi fes més calor.

In [ ]:
URL_ISD = "https://www.ncei.noaa.gov/data/global-hourly/access/{any}/{estacio}.csv"
QUALITAT_DOLENTA = {"2", "3", "6", "7"}      # codis ISD: dada sospitosa o errònia

def baixa_isd_any(any_, estacio, carpeta):
    # Llegeix un any de l'ISD, del Drive si ja hi és; si no, de la NOAA (i el desa)
    cau = Path(carpeta) / f"isd_{estacio}_{any_}.csv"
    if cau.exists():
        return pd.read_csv(cau, usecols=["STATION", "DATE", "NAME", "TMP"], dtype=str)
    r = requests.get(URL_ISD.format(any=any_, estacio=estacio), timeout=120)
    r.raise_for_status()
    d = pd.read_csv(io.StringIO(r.text), usecols=["STATION", "DATE", "NAME", "TMP"], dtype=str)
    d.to_csv(cau, index=False)
    return d

def tmp_a_graus(tmp):
    # '+0215,1' -> 21.5 ; '+9999,9' o qualitat dolenta -> NaN
    valor = tmp.str.split(",").str[0]
    qualitat = tmp.str.split(",").str[1]
    t = pd.to_numeric(valor, errors="coerce") / 10
    dolenta = (valor == "+9999") | qualitat.isin(QUALITAT_DOLENTA)
    return t.where(~dolenta)

# 1. Baixar tots els anys
brut = pd.concat([baixa_isd_any(a, P["estacio_isd"], P["carpeta"]) for a in P["anys"]], ignore_index=True)
nom = brut["NAME"].dropna().iloc[0]
print("Estació:", nom)
assert "BARCELONA" in nom.upper(), f"L'estació baixada no és la del Prat: {nom}. Revisa P['estacio_isd']"

# 2-3. Graus i hora local
brut["temp"] = tmp_a_graus(brut["TMP"])
brut["hora_utc"] = pd.to_datetime(brut["DATE"], utc=True)
brut["hora"] = brut["hora_utc"].dt.tz_convert("Europe/Madrid").dt.tz_localize(None)
print(f"Observacions: {len(brut):,} · sense temperatura vàlida: {brut['temp'].isna().mean():.1%}")

# 4. Una dada per hora
horari = brut.dropna(subset=["temp"]).groupby(brut["hora"].dt.floor("h"))["temp"].mean().rename("temp").to_frame()
horari["data"] = horari.index.normalize()
horari["h"] = horari.index.hour

# 5. Mitjana diària i de cada franja (amb un mínim d'hores per fiar-nos de la mitjana)
FRANJA_DE_HORA = lambda h: f"{(h // 6) * 6:02d}-{(h // 6) * 6 + 6:02d}"
horari["franja"] = horari["h"].map(FRANJA_DE_HORA)
dia = horari.groupby("data")["temp"].agg(["mean", "count"])
prat = dia.loc[dia["count"] >= 20, ["mean"]].rename(columns={"mean": "tmean"})
fr = horari.groupby(["data", "franja"])["temp"].agg(["mean", "count"])
prat_franges = fr.loc[fr["count"] >= 4, "mean"].unstack()
prat = prat.join(prat_franges).reset_index()
prat = prat[prat["data"].dt.year.isin(P["anys"])]
print(f"[OK] Temperatura del Prat: {len(prat)} dies amb mitjana diària vàlida")
print(prat.describe().round(1).loc[["count", "mean", "min", "max"]])

In [ ]:
# Comprovació: el Prat contra ERA5 (la temperatura de l'article 1), si la tenim al Drive
era5_ruta = Path(P["carpeta"]) / "temperatura.csv"
if era5_ruta.exists():
    era5 = pd.read_csv(era5_ruta, parse_dates=["data"])[["data", "tmean"]]
    c = prat.merge(era5, on="data", suffixes=("_prat", "_era5"))
    dif = c["tmean_prat"] - c["tmean_era5"]
    print(f"Prat − ERA5: mitjana {dif.mean():+.2f} °C · correlació {c[['tmean_prat', 'tmean_era5']].corr().iloc[0, 1]:.3f}")
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(c["tmean_era5"], c["tmean_prat"], s=4, alpha=0.4)
    ax.plot([0, 32], [0, 32], color=fs.INK_2, lw=1)
    ax.set_xlabel("ERA5 daily mean (°C)"); ax.set_ylabel("El Prat daily mean (°C)")
    ax.set_title("Reference temperature: airport vs ERA5")
    plt.tight_layout(); plt.savefig("figures_u01/prat_vs_era5.png"); plt.show()

## 3. Consum i qualitat

**Què fa:** llegeix els CSV del consum (Open Data BCN · Datadis) i en treu dues taules:
- **ciutat:** consum diari del sector triat, sumat per a tota la ciutat, en total i per franja;
- **per CP:** consum diari de cada codi postal, per franja.

**Qualitat** (les mateixes regles que l'article 1):
- només **dies complets** (hi són tots els CP actius aquell trimestre);
- fora els dies del **règim B de les franges** (alerta 2) quan es miren franges;
- per CP, només el 2021–2022 (alerta 1: recodificacions de CP des de la tardor del 2023).

**Tipus de dia:** *feiner* = de dilluns a divendres i no festiu; *festiu* = dissabte, diumenge o festiu de Catalunya.

In [ ]:
trossos = []
for f in sorted(Path(P["carpeta"]).glob("*.csv")):
    if "consum_electricitat" in f.name.lower() and any(str(a) in f.name for a in P["anys"]):
        trossos.append(ct.daily_consumption(pd.read_csv(f)))
df = pd.concat(trossos, ignore_index=True)
df = df[df["data"].dt.year.isin(P["anys"])]
if P["sector"] != "Total":
    df = df[df["sector"] == P["sector"]]
FRANGES = sorted(t for t in df["tram"].dropna().unique() if "consta" not in str(t).lower())
NOM_F = {t: f"{int(str(t).split(':')[0][-2:]):02d}-{int(str(t).split(':')[0][-2:]) + 6:02d}" for t in FRANGES}
df = df[df["tram"].isin(FRANGES)].assign(franja=lambda d: d["tram"].map(NOM_F))

# Dies complets i règim de les franges
per_cp_dia = df.groupby(["data", "cp"], as_index=False)["kwh"].sum()
COMPLETS = set(ct.complete_days(per_cp_dia))
ciutat_fr = df.groupby(["data", "franja"])["kwh"].sum().unstack()
_, REGIM_B = ct.band_regime(ciutat_fr, evening="18-24", morning="06-12")

# Tipus de dia
FESTIUS = set(pd.to_datetime(list(holidays.country_holidays("ES", subdiv="CT", years=P["anys"]).keys())))
def tipus_dia(dates):
    return np.where((dates.dt.dayofweek >= 5) | dates.isin(FESTIUS), "festiu", "feiner")

# Taula de la ciutat: un dia per fila, consum en MWh, amb la temperatura del Prat
ciutat = (ciutat_fr.sum(axis=1).rename("total").to_frame().join(ciutat_fr) / 1e3).reset_index()
ciutat = ciutat[ciutat["data"].isin(COMPLETS)].merge(prat, on="data", suffixes=("", "_T"))
ciutat["tipus"] = tipus_dia(ciutat["data"])
ciutat["any"] = ciutat["data"].dt.year
ciutat["regim_B"] = ciutat["data"].isin(REGIM_B)
print(f"[OK] Ciutat: {len(ciutat)} dies complets amb temperatura · règim B: {ciutat['regim_B'].sum()} dies")
print(ciutat.groupby(["any", "tipus"]).size().unstack())

## 4. El núvol de punts de la ciutat: feiners contra caps de setmana i festius

**Què fa:** cada punt és un dia. A l'eix horitzontal hi ha la temperatura mitjana del Prat; a l'eix vertical, el consum diari de la ciutat. Els colors indiquen l'any, de clar (2021) a fosc (2024). La línia és la **mediana** de cada interval d'1 °C: és la «corba en U» sense cap forma imposada.

**Per què separar els tipus de dia:** els caps de setmana la gent és a casa en hores diferents, i el consum de base és diferent. Si es barregen, la U s'eixampla verticalment i se n'amaga la forma.

**Què cal mirar:**
- On és el fons de la U (la **zona de confort**) i si és el mateix als dos panells.
- Si la branca de calor és més empinada que la de fred. La calefacció és sobretot de gas, i per això s'espera que la de fred sigui més suau.
- Si els anys estan desplaçats verticalment (canvis de nivell) o només a la branca de calor (creixement de la resposta, l'article 1).
- **Agost:** a la branca de calor hi ha punts per sota de la corba? Són les vacances: menys gent a la ciutat.
- **Punts molt lluny de la U** (per exemple, dies amb menys de 2.000 MWh): són dies amb dades parcials o dolentes (alerta 5). Aquest notebook no els treu perquè vegis com són. El detector d'anomalies de l'article 1 sí que ho faria.

In [ ]:
def corba_u(x, y, amplada=P["amplada_bin"], min_n=P["min_dies_bin"]):
    # Mediana i quartils de y en intervals de x d'una amplada fixa (la U sense forma imposada)
    t = pd.DataFrame({"x": x, "y": y}).dropna()
    t["bin"] = (np.floor(t["x"] / amplada) + 0.5) * amplada       # centre de l'interval
    g = t.groupby("bin")["y"]
    c = pd.DataFrame({"mediana": g.median(), "q25": g.quantile(0.25), "q75": g.quantile(0.75), "n": g.size()})
    return c[c["n"] >= min_n]

colors = fs.year_colors(P["anys"])
fig, axs = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, tipus, titol in zip(axs, ["feiner", "festiu"], ["Working days", "Weekends and holidays"]):
    d = ciutat[ciutat["tipus"] == tipus]
    for a in P["anys"]:
        da = d[d["any"] == a]
        ax.scatter(da["tmean"], da["total"], s=6, alpha=0.5, color=colors[a], label=str(a))
    c = corba_u(d["tmean"], d["total"])
    ax.plot(c.index, c["mediana"], color=fs.INK, lw=2, label="median (all years)")
    ax.set_xlabel("El Prat daily mean temperature (°C)"); ax.set_title(f"{titol} ({len(d)} days)")
axs[0].set_ylabel(f"{P['sector']} consumption, whole city (MWh/day)"); axs[0].legend(fontsize=8, markerscale=3)
plt.tight_layout(); plt.savefig("figures_u01/nuvol_ciutat.png"); plt.show()

# El fons de la U: a quina temperatura el consum és mínim?
for tipus in ["feiner", "festiu"]:
    c = corba_u(ciutat.loc[ciutat["tipus"] == tipus, "tmean"], ciutat.loc[ciutat["tipus"] == tipus, "total"])
    print(f"{tipus}: mínim de la mediana a {c['mediana'].idxmin():.1f} °C ({c['mediana'].min():,.0f} MWh/dia)")

## 5. Isòcrones: diverses maneres de construir-les

Una isòcrona fixa **el temps** (la franja horària, l'any) i mira la U de tota la ciutat. N'hi ha diverses construccions possibles, i cadascuna respon una pregunta diferent:

| Construcció | Què es fixa | Què varia | Què respon |
|---|---|---|---|
| **5.1** Franja × any | La franja | L'any | La U de cada franja canvia entre anys? (el creixement de l'article 1, franja a franja) |
| **5.2** Franges d'un mateix any, normalitzades | L'any | La franja | Cada franja té la mateixa forma de U? On té el fons cada hora del dia? |
| **5.3** Quina temperatura posar a l'eix | La franja | La temperatura de l'eix | La franja respon a la temperatura del dia, a la de la mateixa franja o als dies anteriors? |

**Compte amb el 2024:** la partició del consum entre franges va canviar el 2024 (alerta 2). Els dies del règim B queden fora de totes les isòcrones per franja.

Per comparar formes i no nivells, a 5.2 cada corba es divideix pel seu consum mitjà a la zona de confort (de 16 a 20 °C): totes valen ≈ 1 al fons de la U.

In [ ]:
# 5.1 Franja × any: una figura per franja, una corba per any. x = temperatura del Prat a la mateixa franja
bon = ciutat[~ciutat["regim_B"] & (ciutat["tipus"] == "feiner")]
fig, axs = plt.subplots(1, 4, figsize=(17, 4.2), sharey=False)
for ax, f in zip(axs, ["00-06", "06-12", "12-18", "18-24"]):
    for a in P["anys"]:
        da = bon[bon["any"] == a]
        c = corba_u(da[f + "_T"], da[f])      # x: temperatura de la franja; y: consum de la franja
        ax.plot(c.index, c["mediana"], "o-", ms=3, color=colors[a], label=str(a))
    ax.set_title(f"Band {f} h"); ax.set_xlabel(f"El Prat temperature, {f} h (°C)")
axs[0].set_ylabel("Consumption in the band (MWh)"); axs[0].legend(fontsize=8)
fig.suptitle("Isochrones 5.1 · same band, different years (working days, band regime A)", y=1.02)
plt.tight_layout(); plt.savefig("figures_u01/isocrones_franja_any.png"); plt.show()

In [ ]:
# 5.2 Les quatre franges d'un mateix any, normalitzades a la zona de confort (16-20 °C del dia)
ANY_52 = 2022
d = bon[bon["any"] == ANY_52]
fig, ax = plt.subplots(figsize=(7, 4.5))
for i, f in enumerate(["00-06", "06-12", "12-18", "18-24"]):
    confort = d.loc[d["tmean"].between(16, 20), f].mean()
    c = corba_u(d["tmean"], d[f] / confort)
    ax.plot(c.index, c["mediana"], "o-", ms=3, color=fs.CAT[i], label=f"{f} h")
ax.axhline(1, color=fs.INK_2, lw=0.8)
ax.set_xlabel("El Prat daily mean temperature (°C)"); ax.set_ylabel("Consumption / comfort-zone mean")
ax.set_title(f"Isochrones 5.2 · the four bands of {ANY_52}, same scale"); ax.legend()
plt.tight_layout(); plt.savefig("figures_u01/isocrones_franges_normalitzades.png"); plt.show()

In [ ]:
# 5.3 La mateixa franja (00-06), amb tres temperatures diferents a l'eix
d = bon.sort_values("data").copy()
d["t_3dies"] = d["tmean"].rolling(3, min_periods=3).mean()       # mitjana dels 3 últims dies (memòria)
opcions = [("tmean", "daily mean"), ("00-06_T", "same band 00-06 h"), ("t_3dies", "mean of last 3 days")]
fig, axs = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
for ax, (col, nom) in zip(axs, opcions):
    for a in P["anys"]:
        da = d[d["any"] == a]
        c = corba_u(da[col], da["00-06"])
        ax.plot(c.index, c["mediana"], "o-", ms=3, color=colors[a], label=str(a))
    ax.set_xlabel(f"El Prat temperature: {nom} (°C)"); ax.set_title(nom)
axs[0].set_ylabel("Consumption 00-06 h (MWh)"); axs[0].legend(fontsize=8)
fig.suptitle("Isochrones 5.3 · which temperature does the night band respond to?", y=1.02)
plt.tight_layout(); plt.savefig("figures_u01/isocrones_eix_temperatura.png"); plt.show()

## 6. Isotropes: la mateixa franja a cada codi postal

**Què fa:** per a la franja triada (`P["franja_isotropes"]`, per defecte de 0 a 6 h, perquè de nit l'illa de calor de l'aire és més forta) dibuixa la U de **cada codi postal** en un petit panell. La U de tota la ciutat surt en gris al fons de cada panell.

**Per poder comparar formes:**
- només dies feiners, complets, del règim A i dels anys 2021–2022 (abans de les recodificacions);
- cada CP es divideix pel seu consum mitjà a la zona de confort (16–20 °C), de manera que totes les corbes valen ≈ 1 al fons i es poden superposar.

**Què cal mirar (la hipòtesi de l'illa de calor):**
- **Un barri més calent que el Prat** hauria de tenir la U **desplaçada cap a l'esquerra**: amb el mateix valor del Prat, a dins hi fa més calor. Consumiria més abans a la branca de calor i menys a la de fred.
- Un barri amb **més aparells** tindria la branca de calor **més alta**, però no desplaçada.
- **Comportament diferent** (engegar l'aire més tard) mouria **només** la branca de calor, no les dues.

Aquí només s'ha de **mirar** si hi ha diferències visibles i de quin tipus. Estimar el desplaçament $\Delta T_i$ ve després, amb el model bayesià.

**CP que s'han de llegir amb compte:** 08007, 08009 i 08011 són gairebé buits per les recodificacions (alerta 1); 08039 i 08040 (port i Zona Franca) tenen molt poc consum residencial. Les seves corbes són soroll.

**CP de referència:** l'estació urbana X4 del Meteocat és al Raval (08001). La seva U és la primera candidata a sortir desplaçada.

In [ ]:
F = P["franja_isotropes"]
per_cp = (df[df["franja"] == F].groupby(["data", "cp"])["kwh"].sum() / 1e3).rename("mwh").reset_index()
per_cp = per_cp[per_cp["data"].dt.year.isin(P["anys_cp"]) & per_cp["data"].isin(COMPLETS) & ~per_cp["data"].isin(REGIM_B)]
per_cp = per_cp.merge(prat[["data", "tmean"]], on="data")
per_cp = per_cp[tipus_dia(per_cp["data"]) == "feiner"]

# Consum de confort de cada CP: la mitjana dels dies entre 16 i 20 °C. Després dividim per aquest valor
confort_cp = per_cp[per_cp["tmean"].between(16, 20)].groupby("cp")["mwh"].mean()
per_cp["rel"] = per_cp["mwh"] / per_cp["cp"].map(confort_cp)
cps = sorted(per_cp["cp"].unique())
print(f"{len(cps)} CP · {per_cp['data'].nunique()} dies feiners de {P['anys_cp']}")

# Corba de tota la ciutat (suma dels CP), normalitzada igual
ciu = per_cp.groupby("data").agg(mwh=("mwh", "sum"), tmean=("tmean", "first")).reset_index()
ciu["rel"] = ciu["mwh"] / ciu.loc[ciu["tmean"].between(16, 20), "mwh"].mean()
c_ciutat = corba_u(ciu["tmean"], ciu["rel"])

ncol = 7
nfil = int(np.ceil(len(cps) / ncol))
fig, axs = plt.subplots(nfil, ncol, figsize=(2.3 * ncol, 2.0 * nfil), sharex=True, sharey=True)
for ax, cp in zip(axs.flat, cps):
    g = per_cp[per_cp["cp"] == cp]
    c = corba_u(g["tmean"], g["rel"])
    ax.plot(c_ciutat.index, c_ciutat["mediana"], color=fs.GRID, lw=3)
    ax.plot(c.index, c["mediana"], color=fs.CAT[1] if cp == "08001" else fs.CAT[0], lw=1.5)
    ax.set_title(cp + (" (X4)" if cp == "08001" else ""), fontsize=8)
    ax.axhline(1, color=fs.INK_2, lw=0.5)
for ax in list(axs.flat)[len(cps):]:
    ax.axis("off")
fig.suptitle(f"Isotropes · band {F} h, each postcode vs the city (grey) · working days {P['anys_cp'][0]}-{P['anys_cp'][-1]} · x = El Prat daily mean (°C)", y=1.0)
plt.tight_layout(); plt.savefig("figures_u01/isotropes_cp.png"); plt.show()

In [ ]:
# Una mesura molt simple, NOMÉS per orientar la mirada (no és l'estimació de ΔT):
# la temperatura del Prat on cada CP arriba a 1,10 vegades el seu consum de confort a la branca de calor.
# Un barri més calent hi hauria d'arribar abans (temperatura més baixa).
def creuament(c, nivell=1.10, desde=20):
    alta = c[c.index >= desde]
    sobre = alta[alta["mediana"] >= nivell]
    return float(sobre.index.min()) if len(sobre) else np.nan

t110 = pd.Series({cp: creuament(corba_u(g["tmean"], g["rel"])) for cp, g in per_cp.groupby("cp")}, name="T_110")
print(f"Ciutat: {creuament(c_ciutat):.1f} °C")
print(t110.sort_values().to_string())
t110.to_csv("figures_u01/creuament_110_per_cp.csv")

## 7. Per comprovar que s'entén (respon-ho tu abans de seguir)

1. Al núvol de la secció 4, per què el fons de la U no és a la mateixa temperatura que el $T_c$ = 20 °C de l'article 1? Pista: a l'article, la temperatura era ERA5 i el model només mirava l'estiu.
2. A les isòcrones 5.3: si la franja de 0 a 6 h respongués només a la temperatura de la mateixa nit, quin dels tres panells hauria de tenir les corbes **més estretes** (menys disperses)? Què surt?
3. A les isotropes: dibuixa en un paper com es veuria la U d'un barri que és 1 K més calent que la ciutat, i la d'un barri amb un 20 % més d'aires condicionats. Quina diferència hi ha entre les dues corbes, i per què la U sencera (amb la branca de fred) és la que permet separar-les?
4. La mesura `T_110` de l'última cel·la barreja l'illa de calor amb alguna altra cosa. Amb quina?